In [3]:
import scipy.io as sio
import numpy as np
from matplotlib.colors import ListedColormap
from scipy.stats import pearsonr
import seaborn as sn
import mne
import matplotlib.pyplot as plt
import copy
from scipy import stats
import pandas as pd
from statsmodels.stats.multitest import fdrcorrection
import copy
import matplotlib.pyplot as plt
import fooof
import specparam
from scipy.stats import permutation_test, false_discovery_control
from fooof import FOOOF
import numpy as np

%matplotlib qt
%gui qt

In [5]:
"""
EEG–Behavioral Correlation Pipeline (FOOOF-flattened relative PSD)
==================================================================

Same as the non-FOOOF correlation pipeline, but each (epoch, channel) PSD is
flattened with FOOOF (initial aperiodic fit only — no peak fitting) BEFORE
the per-epoch relative-power computation.

For each behavioral column:
    - Compute PSD per epoch for Resting (R) and Mindfulness (MF).
    - FOOOF-flatten each (epoch, channel) PSD by subtracting the initial
      aperiodic fit in log10 power:
          init_ap_params = fm._robust_ap_fit(fm.freqs, fm.power_spectrum)
          init_ap_fit    = gen_aperiodic(fm.freqs, init_ap_params)
          flat_PSD       = 10 ** (fm.power_spectrum - init_ap_fit)
    - Compute relative PSD on the flattened spectra with the mean-of-ratios
      definition:
          per epoch: rel_psd[ep, ch, f] = flat_psd[ep, ch, f] / total[ep, ch]
          then mean across epochs.
    - Correlate each (channel, freq) value with the behavioral score using
      scipy.stats.pearsonr with PermutationMethod.
    - Save one topomap figure per multiple-comparison correction method:
        * <col>_pearson_uncorrected.png
        * <col>_pearson_bonferroni.png
        * <col>_pearson_sidak.png
        * <col>_pearson_holm-sidak.png
        * <col>_pearson_holm.png
        * <col>_pearson_simes-hochberg.png
        * <col>_pearson_hommel.png
        * <col>_pearson_fdr_bh.png
        * <col>_pearson_fdr_by.png
        * <col>_pearson_fdr_tsbh.png
        * <col>_pearson_fdr_tsbky.png
        * <col>_pearson_cluster_perm.png
      Each figure is a 2-row x 6-band grid: rows = Resting / Mindfulness.
    - Save:
        * <col>_band_contributions_per_electrode.csv  (and .md)
        * <col>_scatter_resting.png  and  <col>_scatter_mindfulness.png
"""

import os
import warnings
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, PermutationMethod
from statsmodels.stats.multitest import multipletests

# FOOOF / specparam: only the *initial aperiodic fit* is used to flatten the
# spectrum (no peak fitting). Try specparam first; fall back to legacy fooof.
try:
    from fooof import FOOOF
    from fooof.sim.gen import gen_aperiodic
    _FOOOF_BACKEND = "fooof"
except ImportError:  # pragma: no cover
    from specparam import SpectralModel as FOOOF  # type: ignore
    from specparam.sim.gen import gen_aperiodic    # type: ignore
    _FOOOF_BACKEND = "specparam"


# =====================================================================
# CONFIG
# =====================================================================

BEHAVIORAL_FILE = "EEG-Datos conductuales.xlsx"

BEHAVIORAL_COLS = [
    "MAIA",
    "MWQT",
    "Auto-regulación",
    "Confianza",
    "Valencia positiva",
    "Valencia negativa",
    "AR2-MF",
    "AR3-MF",
    "AR4-MF",
    "AR2-AR"
]

# Subjects always excluded (regardless of behavioral column)
ALWAYS_EXCLUDE = ["SUJ_12", "SUJ_33", "SUJ_34", "SUJ_35"]

# Valencia columns: also exclude subjects 30..37
VALENCIA_COLS = ["Valencia positiva", "Valencia negativa"]
VALENCIA_EXTRA_EXCLUDE = [f"SUJ_{i}" for i in range(30, 38)]

ALL_SUBJECT_IDS = list(range(1, 38))  # 1..37

BANDS = {
    "Delta":     (0.5,  4),
    "Theta":     (4,    8),
    "Alpha":     (8,   12),
    "Beta":      (12,  18),
    "Fast Beta": (18,  30),
    "Gamma":     (30,  50),
}

N_PERM = 1000
ALPHA = 0.05
FMIN, FMAX = 0.5, 50.0
PSD_METHOD = "multitaper"

# All correction methods from statsmodels.stats.multitest.multipletests
MULTIPLETESTS_METHODS = [
    "bonferroni",
    "sidak",
    "holm-sidak",
    "holm",
    "simes-hochberg",
    "hommel",
    "fdr_bh",
    "fdr_by",
    "fdr_tsbh",
    "fdr_tsbky",
]

# Cluster-based permutation settings
CLUSTER_N_PERM = 1000
CLUSTER_THRESHOLD_P = 0.05  # primary threshold for forming clusters

# FOOOF: only the initial aperiodic fit is used (no peak fitting), so the
# only relevant kwarg is `aperiodic_mode`. Fit range defaults to (max(FMIN,1.0), FMAX).
FOOOF_DEFAULTS = dict(aperiodic_mode="fixed")
FOOOF_FIT_RANGE = (max(FMIN, 1.0), FMAX)

OUT_DIR = "figures_correlation_fooof"
os.makedirs(OUT_DIR, exist_ok=True)


# =====================================================================
# FOOOF flattening (initial aperiodic fit only, no peak fitting)
# =====================================================================

def _flatten_psd_with_fooof(psd, freqs, fooof_kwargs=None,
                            fit_range=FOOOF_FIT_RANGE, verbose=False):
    """
    Flatten a (n_epochs, n_channels, n_freqs) PSD by removing only the
    initial aperiodic component (no peak fitting).
    """
    fooof_kwargs = {**FOOOF_DEFAULTS, **(fooof_kwargs or {})}
    n_epochs, n_channels, n_freqs = psd.shape
    flat = np.empty_like(psd)

    fmask = (freqs >= fit_range[0]) & (freqs <= fit_range[1])
    if fmask.sum() < 5:
        raise ValueError(
            f"FOOOF fit range {fit_range} contains too few frequency bins "
            f"({fmask.sum()})."
        )

    n_fail = 0
    fm = FOOOF(**fooof_kwargs, verbose=False)

    for ep in range(n_epochs):
        for ch in range(n_channels):
            spec = psd[ep, ch]
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    fm.add_data(freqs, spec, fit_range)
                    init_ap_params = fm._robust_ap_fit(fm.freqs, fm.power_spectrum)
                    init_ap_fit    = gen_aperiodic(fm.freqs, init_ap_params)
                    init_flat_spec = fm.power_spectrum - init_ap_fit
                if np.any(~np.isfinite(init_flat_spec)):
                    raise RuntimeError("non-finite flattened spectrum")
                flat_in = np.power(10.0, init_flat_spec)
                flat[ep, ch] = spec.copy()
                flat[ep, ch, fmask] = flat_in
            except Exception as e:
                n_fail += 1
                flat[ep, ch] = spec
                if verbose:
                    print(f"    FOOOF fail (ep={ep}, ch={ch}): {e}")

    if n_fail and verbose:
        print(f"    FOOOF: {n_fail} / {n_epochs * n_channels} "
              f"spectra fell back to original")
    return flat


# =====================================================================
# HELPERS
# =====================================================================

def get_eeg_filepaths(subject_id):
    return (f"Sujeto{subject_id}_R_processed.fif",
            f"Sujeto{subject_id}_MF_processed.fif")


def subject_id_to_label(subject_id):
    return f"SUJ_{subject_id}"


# =====================================================================
# STEP 1 — Per-subject PSD via mean-of-ratios
# =====================================================================

def compute_relative_psd_for_subject(subject_id, fmin=FMIN, fmax=FMAX):
    """
    Returns rel_R, rel_MF, band_power_R, band_power_MF, total_R, total_MF,
    freqs, info (see original docstring).
    """
    file_R, file_MF = get_eeg_filepaths(subject_id)
    epochs_R  = mne.read_epochs(file_R,  preload=True, verbose=False)
    epochs_MF = mne.read_epochs(file_MF, preload=True, verbose=False)

    psd_R, freqs = epochs_R.compute_psd(
        method=PSD_METHOD, fmin=fmin, fmax=fmax, verbose=False
    ).get_data(return_freqs=True)
    psd_MF, _ = epochs_MF.compute_psd(
        method=PSD_METHOD, fmin=fmin, fmax=fmax, verbose=False
    ).get_data(return_freqs=True)

    psd_R  = _flatten_psd_with_fooof(psd_R,  freqs)
    psd_MF = _flatten_psd_with_fooof(psd_MF, freqs)

    total_R_ep  = np.trapezoid(psd_R,  x=freqs, axis=2)
    total_MF_ep = np.trapezoid(psd_MF, x=freqs, axis=2)
    safe_total_R  = np.where(total_R_ep  > 0, total_R_ep,  np.nan)
    safe_total_MF = np.where(total_MF_ep > 0, total_MF_ep, np.nan)

    band_power_R, band_power_MF = {}, {}
    for band, (lo, hi) in BANDS.items():
        idx = (freqs >= lo) & (freqs <= hi)
        bR_ep  = np.trapezoid(psd_R[:, :, idx],  x=freqs[idx], axis=2)
        bMF_ep = np.trapezoid(psd_MF[:, :, idx], x=freqs[idx], axis=2)
        rel_R_ep  = bR_ep  / safe_total_R
        rel_MF_ep = bMF_ep / safe_total_MF
        band_power_R[band]  = np.nanmean(rel_R_ep,  axis=0)
        band_power_MF[band] = np.nanmean(rel_MF_ep, axis=0)

    rel_R_ep_spec  = psd_R  / safe_total_R[:, :, np.newaxis]
    rel_MF_ep_spec = psd_MF / safe_total_MF[:, :, np.newaxis]
    rel_R  = np.nanmean(rel_R_ep_spec,  axis=0)
    rel_MF = np.nanmean(rel_MF_ep_spec, axis=0)

    total_R  = np.nanmean(safe_total_R,  axis=0)
    total_MF = np.nanmean(safe_total_MF, axis=0)

    return (rel_R, rel_MF, band_power_R, band_power_MF,
            total_R, total_MF, freqs, epochs_R.info)


def build_psd_database(subject_ids):
    print("\n--- Step 1: Computing relative PSD (mean-of-ratios) for all subjects ---")
    raw_db = {}
    info = None

    for sid in subject_ids:
        file_R, file_MF = get_eeg_filepaths(sid)
        if not (os.path.exists(file_R) and os.path.exists(file_MF)):
            print(f"  [skip] Subject {sid}: missing FIF file(s)")
            continue
        try:
            (rel_R, rel_MF, bp_R, bp_MF,
             tot_R, tot_MF, fr, inf) = compute_relative_psd_for_subject(sid)
        except Exception as e:
            print(f"  [error] Subject {sid}: {e}")
            continue

        raw_db[sid] = {
            "R": rel_R, "MF": rel_MF, "freqs": fr,
            "band_power_R": bp_R, "band_power_MF": bp_MF,
            "total_R": tot_R, "total_MF": tot_MF,
        }
        if info is None:
            info = inf
        print(f"  [ok]   Subject {sid}  (n_freqs={len(fr)})")

    if not raw_db:
        return {}, None, None

    common_freqs = None
    for d in raw_db.values():
        common_freqs = d["freqs"] if common_freqs is None else np.intersect1d(common_freqs, d["freqs"])
    print(f"  Common grid: {len(common_freqs)} bins "
          f"({common_freqs.min():.2f}–{common_freqs.max():.2f} Hz)")

    psd_db = {}
    for sid, d in raw_db.items():
        idx = np.isin(d["freqs"], common_freqs)
        psd_db[sid] = {
            "R":  d["R"][:,  idx],
            "MF": d["MF"][:, idx],
            "band_power_R":  d["band_power_R"],
            "band_power_MF": d["band_power_MF"],
            "total_R":  d["total_R"],
            "total_MF": d["total_MF"],
        }

    return psd_db, common_freqs, info


# =====================================================================
# STEP 2 — Correlations
# =====================================================================

def correlate_with_behavior(data, responses, method):
    """
    data:      (n_subjects, n_channels, n_freqs)
    responses: (n_subjects,)
    Returns r_vals, p_vals -> (n_channels, n_freqs).
    """
    n_subj, n_ch, n_f = data.shape
    r_vals = np.full((n_ch, n_f), np.nan)
    p_vals = np.full((n_ch, n_f), np.nan)

    for ch in range(n_ch):
        for f in range(n_f):
            x = data[:, ch, f]
            y = responses
            if np.any(np.isnan(x)) or np.any(np.isnan(y)):
                continue
            res = pearsonr(x, y, method=method)
            r_vals[ch, f] = res.statistic
            p_vals[ch, f] = res.pvalue
    return r_vals, p_vals


def correlate_band_power_with_behavior(psd_db, subject_ids, responses,
                                       condition_key, method):
    """
    Correlate each band's RELATIVE POWER against the behavioral score --
    one Pearson r/p per (band, channel). "Relative power" here is exactly
    band_power_R / band_power_MF as already computed in
    compute_relative_psd_for_subject: per epoch, sum (trapezoidal integral
    of) the PSD across the band's frequencies and divide by that epoch's
    total PSD, then average across epochs. This is the same quantity used
    in the band-contribution table and the scatter plots -- this function
    just runs the actual (band, channel)-level significance test on it.

    condition_key : "R" or "MF"
    method        : passed straight to scipy.stats.pearsonr (e.g. a
                     PermutationMethod instance, for consistency with the
                     bin-wise correlate_with_behavior).

    Returns
    -------
    r_band, p_band : dict {band_name: (n_channels,) array}
    """
    bp_key = "band_power_R" if condition_key == "R" else "band_power_MF"
    r_band, p_band = {}, {}
    for band in BANDS:
        x_stack = np.stack([psd_db[s][bp_key][band] for s in subject_ids])  # (n_subj, n_ch)
        n_ch = x_stack.shape[1]
        r_vals = np.full(n_ch, np.nan)
        p_vals = np.full(n_ch, np.nan)
        for ch in range(n_ch):
            x = x_stack[:, ch]
            if np.any(np.isnan(x)) or np.any(np.isnan(responses)):
                continue
            res = pearsonr(x, responses, method=method)
            r_vals[ch] = res.statistic
            p_vals[ch] = res.pvalue
        r_band[band] = r_vals
        p_band[band] = p_vals
    return r_band, p_band


def apply_multipletests(p_vals, method, alpha=ALPHA):
    """
    Apply a statsmodels.multipletests correction across all non-NaN entries
    of p_vals (flattened). Returns corrected p-values reshaped to p_vals.shape.
    """
    flat = p_vals.flatten()
    valid = ~np.isnan(flat)
    p_corr = np.full_like(flat, np.nan, dtype=float)
    if valid.sum() > 0:
        _, p_c, _, _ = multipletests(flat[valid], alpha=alpha, method=method)
        p_corr[valid] = p_c
    return p_corr.reshape(p_vals.shape)


def apply_multipletests_band(p_band, method, alpha=ALPHA):
    """
    Same correction as apply_multipletests, but for the band-level
    {band_name: (n_channels,) p-values} dict structure. All bands x
    channels for one condition are corrected JOINTLY (one family of
    len(BANDS) * n_channels tests), matching how the bin-wise p_R / p_MF
    are corrected as a single flattened block. Returns a dict with the
    same keys/shapes as `p_band`.
    """
    bands = list(p_band.keys())
    sizes = [p_band[b].shape[0] for b in bands]
    flat = np.concatenate([p_band[b] for b in bands])
    valid = ~np.isnan(flat)
    p_corr_flat = np.full_like(flat, np.nan, dtype=float)
    if valid.sum() > 0:
        _, p_c, _, _ = multipletests(flat[valid], alpha=alpha, method=method)
        p_corr_flat[valid] = p_c
    out, idx = {}, 0
    for b, n in zip(bands, sizes):
        out[b] = p_corr_flat[idx:idx + n]
        idx += n
    return out


# =====================================================================
# STEP 2b — Save r / p-values (uncorrected + every correction) to Excel
# =====================================================================

def save_r_p_values_to_excel(col_name, freqs, ch_names,
                              r_R, p_R, p_R_corr_by_method, r_R_cl, p_R_cl,
                              r_MF, p_MF, p_MF_corr_by_method, r_MF_cl, p_MF_cl,
                              r_band_R, p_band_R, p_band_R_corr_by_method,
                              r_band_MF, p_band_MF, p_band_MF_corr_by_method,
                              out_dir=OUT_DIR):
    """
    Save every Pearson r and p-value to a single Excel file with two sheets:

        <col_name>_r_p_values.xlsx

    Sheet "band_power_r_p_values" (band-level -- THIS is what the
    <col>_pearson_<method>.png topomap figures are now built from):
    one row per (electrode, band, condition), Pearson r/p computed directly
    on each band's RELATIVE POWER (sum of the PSD across the band's
    frequencies divided by the total PSD, averaged per subject -- i.e.
    band_power_R / band_power_MF), for uncorrected and every multipletests
    correction:
        Condition, Band, Electrode, r, p_uncorrected,
        p_bonferroni, p_sidak, p_holm-sidak, p_holm, p_simes-hochberg,
        p_hommel, p_fdr_bh, p_fdr_by, p_fdr_tsbh, p_fdr_tsbky

    Sheet "freq_bin_r_p_values" (bin-wise, full spectral resolution):
    one row per (electrode, frequency bin, condition), Pearson r/p for
    uncorrected, every multipletests correction, and the cluster-permutation
    correction (which needs the full frequency resolution to find
    contiguous clusters, so it isn't part of the band-level sheet):
        Condition, Electrode, Freq_Hz, r, p_uncorrected,
        p_bonferroni, ..., p_fdr_tsbky, p_cluster_perm, r_cluster_perm

    In both sheets, `r` is the pointwise Pearson r shared by p_uncorrected
    and every multipletests column (correction only changes p, never r).
    `r_cluster_perm` is kept as its own column because the cluster-based
    test computes r via a separate vectorized path (cluster_permutation_pearson)
    — it should match `r` wherever both are defined, but NaN-handling can
    differ slightly between the two code paths, so both are kept for
    transparency/auditability.
    """
    def _long_frame_bins(r_vals, p_vals, p_corr_by_method, r_cl, p_cl, condition):
        n_ch, n_f = r_vals.shape
        ch_grid, f_grid = np.meshgrid(np.arange(n_ch), np.arange(n_f), indexing="ij")
        data = {
            "Condition":     condition,
            "Electrode":     np.asarray(ch_names)[ch_grid.ravel()],
            "Freq_Hz":       freqs[f_grid.ravel()],
            "r":             r_vals.ravel(),
            "p_uncorrected": p_vals.ravel(),
        }
        for method in MULTIPLETESTS_METHODS:
            data[f"p_{method}"] = p_corr_by_method[method].ravel()
        data["p_cluster_perm"] = p_cl.ravel()
        data["r_cluster_perm"] = r_cl.ravel()
        return pd.DataFrame(data)

    def _long_frame_bands(r_band_vals, p_band_vals, p_band_corr_by_method, condition):
        frames = []
        for band in BANDS:
            r_arr = r_band_vals[band]
            n_ch = r_arr.shape[0]
            data = {
                "Condition":     [condition] * n_ch,
                "Band":          [band] * n_ch,
                "Electrode":     list(ch_names),
                "r":             r_arr,
                "p_uncorrected": p_band_vals[band],
            }
            for method in MULTIPLETESTS_METHODS:
                data[f"p_{method}"] = p_band_corr_by_method[method][band]
            frames.append(pd.DataFrame(data))
        return pd.concat(frames, ignore_index=True)

    df_bins_R  = _long_frame_bins(r_R,  p_R,  p_R_corr_by_method,  r_R_cl,  p_R_cl,  "Resting")
    df_bins_MF = _long_frame_bins(r_MF, p_MF, p_MF_corr_by_method, r_MF_cl, p_MF_cl, "Mindfulness")
    df_bins = pd.concat([df_bins_R, df_bins_MF], ignore_index=True)

    df_bands_R  = _long_frame_bands(r_band_R,  p_band_R,  p_band_R_corr_by_method,  "Resting")
    df_bands_MF = _long_frame_bands(r_band_MF, p_band_MF, p_band_MF_corr_by_method, "Mindfulness")
    df_bands = pd.concat([df_bands_R, df_bands_MF], ignore_index=True)

    fname = os.path.join(out_dir, f"{col_name.replace(' ', '_')}_r_p_values.xlsx")
    with pd.ExcelWriter(fname, engine="openpyxl") as writer:
        df_bands.to_excel(writer, sheet_name="band_power_r_p_values", index=False)
        df_bins.to_excel(writer, sheet_name="freq_bin_r_p_values", index=False)
        for sheet_name in ("band_power_r_p_values", "freq_bin_r_p_values"):
            ws = writer.sheets[sheet_name]
            ws.freeze_panes = "A2"  # keep the header row visible while scrolling
            for col_cells in ws.columns:
                width = max((len(str(c.value)) if c.value is not None else 0)
                            for c in col_cells) + 2
                ws.column_dimensions[col_cells[0].column_letter].width = min(width, 22)

    print(f"  saved: {fname}")
    return fname


# =====================================================================
# STEP 2c — Cluster-based permutation on Pearson r
# =====================================================================

def _pearson_r_along_subjects(data, y):
    """
    Vectorized Pearson r between each (channel, freq) entry of `data` and
    the 1-D vector `y`.
        data: (n_subjects, n_channels, n_freqs)
        y:    (n_subjects,)
    Returns r: (n_channels, n_freqs), with NaN where variance is 0.
    """
    n = y.shape[0]
    y_c = y - y.mean()
    x_c = data - data.mean(axis=0, keepdims=True)
    num = np.einsum("i,ijk->jk", y_c, x_c)
    den = np.sqrt((x_c ** 2).sum(axis=0) * (y_c ** 2).sum())
    with np.errstate(invalid="ignore", divide="ignore"):
        r = np.where(den > 0, num / den, np.nan)
    return r


def cluster_permutation_pearson(data, responses, info, freqs,
                                n_perm=CLUSTER_N_PERM,
                                threshold_p=CLUSTER_THRESHOLD_P,
                                alpha=ALPHA, seed=0):
    """
    Cluster-based permutation test on Pearson correlations between each
    (channel, freq) entry of `data` and `responses`, using MNE's spatial
    sensor adjacency and tail=0 (two-sided).

    Returns
    -------
    r_obs       : (n_channels, n_freqs) observed Pearson r
    p_corr      : (n_channels, n_freqs) cluster-corrected p-values
                   (each entry within a significant cluster gets that
                   cluster's p-value; entries outside any cluster get 1.0).
    cluster_info: dict with keys 'clusters', 'cluster_pv', 't_obs',
                  'threshold' (returned for completeness/debugging).
    """
    from scipy.stats import t as student_t
    from mne.stats import permutation_cluster_test
    from mne.channels import find_ch_adjacency

    n_subj = data.shape[0]
    rng = np.random.default_rng(seed)

    # Observed r and primary threshold (two-sided) translated from a
    # critical r at threshold_p with n_subj-2 df via |t| = r*sqrt((n-2)/(1-r^2)).
    r_obs = _pearson_r_along_subjects(data, responses)
    df = n_subj - 2
    if df <= 0:
        return r_obs, np.ones_like(r_obs), {"clusters": [], "cluster_pv": [],
                                            "t_obs": None, "threshold": None}
    t_crit = student_t.ppf(1 - threshold_p / 2.0, df)
    r_crit = t_crit / np.sqrt(t_crit ** 2 + df)

    # Build sensor adjacency for the topographic dimension.
    adjacency, _ = find_ch_adjacency(info, ch_type="eeg")

    # Use a permutation-of-r approach by passing two "groups" of single-subject
    # r-maps generated by signed permutation of `responses`. We construct a
    # test statistic by re-correlating with permuted responses and pulling
    # everything through mne.stats.permutation_cluster_test on the r-maps.
    #
    # The cleanest equivalent: run our own permutations of `responses`,
    # compute r-maps, threshold at |r| > r_crit, cluster via MNE's adjacency,
    # build a null of max cluster mass.

    from mne.stats.cluster_level import _find_clusters

    # Observed clusters on |r_obs| > r_crit, signed cluster mass = sum(r) in cluster.
    obs_stats = r_obs.copy()
    obs_stats[~np.isfinite(obs_stats)] = 0.0

    # _find_clusters expects a 1-D stat and a connectivity. Flatten (chan*freq)
    # and use a Kronecker-style adjacency: connect samples that share a
    # frequency bin AND are spatial neighbors, OR share a channel AND are
    # adjacent in frequency.
    n_ch, n_f = r_obs.shape

    def _flat_idx(ch, f):
        return ch * n_f + f

    # Build sparse adjacency on the flattened stat.
    from scipy import sparse
    rows, cols = [], []
    adj_dense = adjacency.toarray().astype(bool)
    for ch in range(n_ch):
        # spatial neighbors at the same freq
        for ch2 in np.where(adj_dense[ch])[0]:
            if ch2 == ch:
                continue
            for f in range(n_f):
                rows.append(_flat_idx(ch, f))
                cols.append(_flat_idx(ch2, f))
        # frequency neighbors at the same channel
        for f in range(n_f - 1):
            rows.append(_flat_idx(ch, f))
            cols.append(_flat_idx(ch, f + 1))
            rows.append(_flat_idx(ch, f + 1))
            cols.append(_flat_idx(ch, f))
    flat_adj = sparse.csr_matrix(
        (np.ones(len(rows), dtype=bool), (rows, cols)),
        shape=(n_ch * n_f, n_ch * n_f),
    )

    obs_flat = obs_stats.flatten()
    clusters_obs, stats_obs = _find_clusters(
        obs_flat, threshold=r_crit, tail=0, adjacency=flat_adj,
    )

    if len(clusters_obs) == 0:
        return r_obs, np.ones_like(r_obs), {
            "clusters": [], "cluster_pv": [],
            "t_obs": r_obs, "threshold": r_crit,
        }

    # Null distribution: max |cluster mass| under random permutations of y.
    null_max = np.empty(n_perm)
    for i in range(n_perm):
        y_perm = rng.permutation(responses)
        r_perm = _pearson_r_along_subjects(data, y_perm)
        r_perm[~np.isfinite(r_perm)] = 0.0
        _, perm_stats = _find_clusters(
            r_perm.flatten(), threshold=r_crit, tail=0, adjacency=flat_adj,
        )
        null_max[i] = np.max(np.abs(perm_stats)) if len(perm_stats) > 0 else 0.0

    # p-value per observed cluster: P(null_max >= |obs_stat|).
    cluster_pv = np.array([
        (np.sum(null_max >= abs(s)) + 1) / (n_perm + 1)
        for s in stats_obs
    ])

    # Build a p-value map: entries in each significant cluster get that
    # cluster's p-value; others get 1.0.
    p_corr_flat = np.ones(n_ch * n_f, dtype=float)
    for clu, pv in zip(clusters_obs, cluster_pv):
        # _find_clusters can return either index arrays or boolean masks.
        if isinstance(clu, tuple):
            idx = clu[0]
        elif isinstance(clu, np.ndarray) and clu.dtype == bool:
            idx = np.where(clu)[0]
        else:
            idx = np.asarray(clu)
        # Only paint significant clusters as significant, but always store
        # the actual cluster p-value so the user can also see near-misses.
        p_corr_flat[idx] = np.minimum(p_corr_flat[idx], pv)

    p_corr = p_corr_flat.reshape(n_ch, n_f)
    return r_obs, p_corr, {
        "clusters": clusters_obs,
        "cluster_pv": cluster_pv,
        "t_obs": r_obs,
        "threshold": r_crit,
    }


# =====================================================================
# STEP 3 — Band-contribution table (unchanged)
# =====================================================================

def band_contribution_table(psd_db, subject_ids, freqs, ch_names, aggregate="mean"):
    assert aggregate in ("mean", "median")
    agg = np.nanmean if aggregate == "mean" else np.nanmedian
    bands = list(BANDS.keys())
    states = ["Rest", "MF"]

    bp_R_stack  = {b: np.stack([psd_db[s]["band_power_R"][b]  for s in subject_ids])
                   for b in bands}
    bp_MF_stack = {b: np.stack([psd_db[s]["band_power_MF"][b] for s in subject_ids])
                   for b in bands}

    band_columns, column_labels, cols = [], [], []
    for band in bands:
        for state in states:
            arr = bp_R_stack[band] if state == "Rest" else bp_MF_stack[band]
            cols.append(agg(arr, axis=0))
            band_columns.append((band, state))
            column_labels.append(f"{band}_{state}")
    values = np.stack(cols, axis=1)

    union_lo = min(lo for (lo, _) in BANDS.values())
    union_hi = max(hi for (_, hi) in BANDS.values())
    idx = (freqs >= union_lo) & (freqs <= union_hi)
    f_sub = freqs[idx]

    def sum_over_union(state):
        key = "R" if state == "Rest" else "MF"
        psd_sub = np.stack([psd_db[s][key][:, idx] for s in subject_ids])
        per_subj = np.trapezoid(psd_sub, x=f_sub, axis=2)
        return agg(per_subj, axis=0)

    sum_rest = sum_over_union("Rest")[:, None]
    sum_mf   = sum_over_union("MF")[:, None]
    delta    = sum_mf - sum_rest

    values_full = np.concatenate([values, sum_rest, sum_mf, delta], axis=1)
    column_labels_full = column_labels + ["Sum_Rest", "Sum_MF", "DeltaSum_MF-Rest"]

    return {
        "electrode":      np.array(ch_names),
        "columns":        column_labels_full,
        "values":         values_full,
        "band_columns":   band_columns,
        "sum_freq_range": (float(union_lo), float(union_hi)),
        "aggregate":      aggregate,
    }


def save_band_contribution_table(table, out_path, fmt="csv", float_fmt="%.6f"):
    assert fmt in ("csv", "tsv", "markdown")
    header = ["electrode"] + table["columns"]
    if fmt in ("csv", "tsv"):
        sep = "," if fmt == "csv" else "\t"
        lines = [sep.join(header)]
        for i, ch in enumerate(table["electrode"]):
            row = [ch] + [float_fmt % v for v in table["values"][i]]
            lines.append(sep.join(row))
    else:
        widths = [max(len(h), 10) for h in header]
        def fmt_row(cells):
            return "| " + " | ".join(c.ljust(w) for c, w in zip(cells, widths)) + " |"
        sep_row = "| " + " | ".join("-" * w for w in widths) + " |"
        lines = [fmt_row(header), sep_row]
        for i, ch in enumerate(table["electrode"]):
            row = [ch] + [float_fmt % v for v in table["values"][i]]
            lines.append(fmt_row(row))
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
        f.write("\n")
    return os.path.abspath(out_path)


# =====================================================================
# STEP 4 — Plotting (now 2 rows: Resting, Mindfulness; no Cohen's d row)
# =====================================================================

def _plot_topomap_grid(results_by_condition, freqs, info, correction_label,
                       behavioral_name, alpha=ALPHA):
    """
    Plot a 2-row x 6-band topomap grid for one correction method.
        rows: Resting, Mindfulness
        cols: 6 frequency bands
    A channel is marked significant (black dot) if any frequency bin within
    the band has p < alpha after the chosen correction.
    """
    row_labels = [
        ("R",  "Resting"),
        ("MF", "Mindfulness"),
    ]
    n_rows = len(row_labels)
    n_cols = len(BANDS)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3.4 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    all_r = np.concatenate([
        results_by_condition[k]["r_vals"].flatten() for k in results_by_condition
    ])
    vmax = np.nanmax(np.abs(all_r))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    vlim = (-vmax, vmax)

    im = None
    for row_i, (cond_key, cond_label) in enumerate(row_labels):
        res = results_by_condition[cond_key]
        r_vals = res["r_vals"]
        p_use = res["p_corr"]

        for col_i, (band_name, (fmin_b, fmax_b)) in enumerate(BANDS.items()):
            ax = axes[row_i, col_i]
            band_idx = (freqs >= fmin_b) & (freqs <= fmax_b)
            if not band_idx.any():
                ax.set_visible(False)
                continue

            r_band = np.nanmean(r_vals[:, band_idx], axis=1)
            with np.errstate(invalid="ignore"):
                sig_mat = (p_use[:, band_idx] < alpha)
            mask = sig_mat.any(axis=1)

            im, _ = mne.viz.plot_topomap(
                r_band, info, axes=ax, cmap="jet",
                vlim=vlim, show=False, mask=mask,
                mask_params=dict(marker="o", markerfacecolor="black",
                                 markeredgecolor="white", markersize=8),
            )
            if row_i == 0:
                ax.set_title(band_name, fontsize=11)
            if col_i == 0:
                ax.set_ylabel(cond_label, fontsize=11, rotation=90, labelpad=20)

    fig.suptitle(
        f"{behavioral_name} — Pearson r — {correction_label} (alpha={alpha})",
        fontsize=13,
    )

    if im is not None:
        cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7, location="right")
        cbar.set_label("Pearson r")

    plt.subplots_adjust(left=0.06, right=0.92, top=0.90, bottom=0.05,
                        wspace=0.25, hspace=0.30)

    safe_label = correction_label.replace(" ", "_").replace("/", "_")
    fname = os.path.join(
        OUT_DIR,
        f"{behavioral_name.replace(' ', '_')}_pearson_{safe_label}.png",
    )
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {fname}")


def _plot_topomap_grid_bandpower(results_by_condition, info, correction_label,
                                 behavioral_name, alpha=ALPHA):
    """
    Plot a 2-row x 6-band topomap grid for one correction method, using
    Pearson correlations computed directly on each band's RELATIVE POWER
    (sum of PSD across the band's frequencies divided by total PSD,
    averaged per subject -- band_power_R / band_power_MF), NOT a proxy
    derived from bin-wise significance.

        rows: Resting, Mindfulness
        cols: 6 frequency bands
    A channel is marked significant (black dot) iff the band-level p-value
    (after the chosen correction) is < alpha -- i.e. the relative power in
    that band, at that electrode, is itself significantly correlated with
    the behavioral score.

    results_by_condition: dict with keys "R", "MF", each holding
        {"r_band": {band_name: (n_channels,) r}, "p_band": {band_name: (n_channels,) p}}
    """
    row_labels = [
        ("R",  "Resting"),
        ("MF", "Mindfulness"),
    ]
    n_rows = len(row_labels)
    n_cols = len(BANDS)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3.4 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    all_r = np.concatenate([
        np.concatenate(list(results_by_condition[k]["r_band"].values()))
        for k in results_by_condition
    ])
    vmax = np.nanmax(np.abs(all_r))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    vlim = (-vmax, vmax)

    im = None
    for row_i, (cond_key, cond_label) in enumerate(row_labels):
        res = results_by_condition[cond_key]

        for col_i, band_name in enumerate(BANDS.keys()):
            ax = axes[row_i, col_i]
            r_band = res["r_band"][band_name]
            p_band = res["p_band"][band_name]
            with np.errstate(invalid="ignore"):
                mask = p_band < alpha

            im, _ = mne.viz.plot_topomap(
                r_band, info, axes=ax, cmap="jet",
                vlim=vlim, show=False, mask=mask,
                mask_params=dict(marker="o", markerfacecolor="black",
                                 markeredgecolor="white", markersize=8),
            )
            if row_i == 0:
                ax.set_title(band_name, fontsize=11)
            if col_i == 0:
                ax.set_ylabel(cond_label, fontsize=11, rotation=90, labelpad=20)

    fig.suptitle(
        f"{behavioral_name} — Pearson r (band relative power) — "
        f"{correction_label} (alpha={alpha})",
        fontsize=13,
    )

    if im is not None:
        cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7, location="right")
        cbar.set_label("Pearson r")

    plt.subplots_adjust(left=0.06, right=0.92, top=0.90, bottom=0.05,
                        wspace=0.25, hspace=0.30)

    safe_label = correction_label.replace(" ", "_").replace("/", "_")
    fname = os.path.join(
        OUT_DIR,
        f"{behavioral_name.replace(' ', '_')}_pearson_{safe_label}.png",
    )
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {fname}")


def _plot_electrode_scatter_grid(psd_db, subject_ids, responses,
                                 ch_names, behavioral_name, condition_key,
                                 condition_label, r_band, p_band,
                                 top_k=6, max_per_band=2):
    """
    r_band, p_band: dicts {band_name: (n_channels,) array} -- the SAME
    band-level relative-power correlation results (uncorrected r/p from
    correlate_band_power_with_behavior) used to drive the topomap
    significance masks, so the r/p shown in scatter titles match what's
    plotted elsewhere instead of being recomputed with a different p-value
    method.
    """
    bands = list(BANDS.keys())
    bp_key = "band_power_R" if condition_key == "R" else "band_power_MF"

    bp_stack = {
        b: np.stack([psd_db[s][bp_key][b] for s in subject_ids])
        for b in bands
    }
    n_subj, n_ch = bp_stack[bands[0]].shape

    candidates = []
    for b in bands:
        for ch in range(n_ch):
            r, p = r_band[b][ch], p_band[b][ch]
            if not np.isfinite(r) or not np.isfinite(p):
                continue
            x = bp_stack[b][:, ch]
            candidates.append((b, ch, float(r), float(p), x))

    if not candidates:
        print(f"  [warn] No valid scatter candidates for {behavioral_name}/{condition_label}")
        return

    by_band = {b: [] for b in bands}
    for c in candidates:
        by_band[c[0]].append(c)
    picked = []
    for b in bands:
        s = sorted(by_band[b], key=lambda t: -abs(t[2]))[:max_per_band]
        picked.extend(s)
    picked = sorted(picked, key=lambda t: -abs(t[2]))[:top_k]

    if not picked:
        return

    n = len(picked)
    n_cols = min(3, n)
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4.0 * n_cols, 3.5 * n_rows),
                             squeeze=False)
    fig.suptitle(
        f"{behavioral_name} — top electrode×band correlations "
        f"({condition_label})\n"
        f"x: per-subject mean-of-ratios relative band power · "
        f"y: {behavioral_name}",
        fontsize=12,
    )

    for i, (band, ch, r, p, x) in enumerate(picked):
        ax = axes[i // n_cols, i % n_cols]
        ax.scatter(x, responses, color="#444444", s=30, alpha=0.85, zorder=3)

        if len(np.unique(x)) > 1:
            slope, intercept = np.polyfit(x, responses, 1)
            xs = np.linspace(x.min(), x.max(), 50)
            ax.plot(xs, slope * xs + intercept, color="#B6464A",
                    linewidth=1.5, alpha=0.85, zorder=2)

        ax.set_title(f"{ch_names[ch]} · {band}\nr = {r:+.2f}, p = {p:.3f}",
                     fontsize=10)
        ax.set_xlabel(f"{band} rel. power")
        ax.set_ylabel(behavioral_name)
        ax.grid(alpha=0.25, linewidth=0.5)

    for j in range(n, n_rows * n_cols):
        axes[j // n_cols, j % n_cols].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fname = os.path.join(
        OUT_DIR,
        f"{behavioral_name.replace(' ', '_')}_scatter_{condition_label.lower().replace(' ', '_')}.png",
    )
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {fname}")


# =====================================================================
# MAIN PIPELINE
# =====================================================================

def load_behavioral_table(path):
    return pd.read_excel(path)


def get_subject_selection_for_column(df, col_name):
    sub_col = df["Sujeto"]
    val_col = df[col_name]
    mask = sub_col.notna() & val_col.notna()
    excludes = list(ALWAYS_EXCLUDE)
    if col_name in VALENCIA_COLS:
        excludes += VALENCIA_EXTRA_EXCLUDE
    mask &= ~sub_col.isin(excludes)

    sub_clean = sub_col[mask].astype(str).tolist()
    val_clean = val_col[mask].to_numpy()

    subject_ids, responses = [], []
    for label, value in zip(sub_clean, val_clean):
        try:
            sid = int(label.replace("SUJ_", "").strip())
        except ValueError:
            print(f'  [warn] Could not parse subject label "{label}" — skipped.')
            continue
        subject_ids.append(sid)
        responses.append(float(value))
    return subject_ids, np.array(responses, dtype=float)


def run_for_behavioral_column(df, col_name, psd_db, freqs, info):
    print("\n" + "=" * 60)
    print(f"Behavioral column: {col_name}")
    print("=" * 60)

    subject_ids, responses = get_subject_selection_for_column(df, col_name)
    sub_with_eeg = [s for s in subject_ids if s in psd_db]
    keep_idx = [i for i, s in enumerate(subject_ids) if s in psd_db]
    responses = responses[keep_idx]

    print(f"  N subjects after exclusions + EEG availability: {len(sub_with_eeg)}")
    print(f"  Subjects used: {sub_with_eeg}")
    if len(sub_with_eeg) < 4:
        print(f"  [warn] Too few subjects ({len(sub_with_eeg)}) — skipping.")
        return

    # ---- Band-contribution table (saved per behavioral column) ----
    ch_names = info["ch_names"]
    table = band_contribution_table(psd_db, sub_with_eeg, freqs, ch_names)
    csv_path = os.path.join(
        OUT_DIR,
        f"{col_name.replace(' ', '_')}_band_contributions_per_electrode.csv",
    )
    md_path = csv_path.replace(".csv", ".md")
    save_band_contribution_table(table, csv_path, fmt="csv")
    save_band_contribution_table(table, md_path,  fmt="markdown")
    print(f"  band-contribution table: {csv_path}")
    print(f"                              {md_path}")

    # ---- Correlation pipeline (only R and MF; no Cohen's d row) ----
    psd_R  = np.stack([psd_db[s]["R"]  for s in sub_with_eeg])
    psd_MF = np.stack([psd_db[s]["MF"] for s in sub_with_eeg])

    method = PermutationMethod(n_resamples=N_PERM)

    # ---- Band-level correlation: Pearson r/p on each band's RELATIVE POWER
    #      (sum of PSD across the band / total PSD, averaged per subject --
    #      band_power_R / band_power_MF). THIS drives the significance
    #      markers in the <col>_pearson_<method>.png topomap figures below. ----
    print("  -> Correlating band relative power (Resting)...")
    r_band_R,  p_band_R  = correlate_band_power_with_behavior(
        psd_db, sub_with_eeg, responses, "R", method=method,
    )
    print("  -> Correlating band relative power (Mindfulness)...")
    r_band_MF, p_band_MF = correlate_band_power_with_behavior(
        psd_db, sub_with_eeg, responses, "MF", method=method,
    )

    # ---- Figure 1: uncorrected (band relative power) ----
    results_uncorr_band = {
        "R":  dict(r_band=r_band_R,  p_band=p_band_R),
        "MF": dict(r_band=r_band_MF, p_band=p_band_MF),
    }
    _plot_topomap_grid_bandpower(results_uncorr_band, info, "uncorrected", col_name)

    # ---- Figures 2..11: each multipletests correction (band relative power) ----
    p_band_R_corr_by_method  = {}
    p_band_MF_corr_by_method = {}
    for mt_method in MULTIPLETESTS_METHODS:
        p_band_R_c  = apply_multipletests_band(p_band_R,  mt_method)
        p_band_MF_c = apply_multipletests_band(p_band_MF, mt_method)
        p_band_R_corr_by_method[mt_method]  = p_band_R_c
        p_band_MF_corr_by_method[mt_method] = p_band_MF_c
        results_c_band = {
            "R":  dict(r_band=r_band_R,  p_band=p_band_R_c),
            "MF": dict(r_band=r_band_MF, p_band=p_band_MF_c),
        }
        _plot_topomap_grid_bandpower(results_c_band, info, mt_method, col_name)

    # ---- Bin-wise (full spectral resolution) correlation -- kept only to
    #      feed the cluster-permutation test below (which needs frequency
    #      resolution to find contiguous clusters) and the fine-grained
    #      "freq_bin_r_p_values" Excel sheet; no longer drives its own
    #      topomap figures (see band-level section above). ----
    print("  -> Correlating Resting condition (bin-wise, for cluster-perm)...")
    r_R,  p_R  = correlate_with_behavior(psd_R,  responses, method=method)
    print("  -> Correlating Mindfulness condition (bin-wise, for cluster-perm)...")
    r_MF, p_MF = correlate_with_behavior(psd_MF, responses, method=method)

    p_R_corr_by_method  = {mt: apply_multipletests(p_R,  mt) for mt in MULTIPLETESTS_METHODS}
    p_MF_corr_by_method = {mt: apply_multipletests(p_MF, mt) for mt in MULTIPLETESTS_METHODS}

    # ---- Figure 12: cluster-based permutation on Pearson r ----
    print("  -> Cluster-based permutation (Resting)...")
    r_R_cl, p_R_cl, _ = cluster_permutation_pearson(
        psd_R, responses, info, freqs,
    )
    print("  -> Cluster-based permutation (Mindfulness)...")
    r_MF_cl, p_MF_cl, _ = cluster_permutation_pearson(
        psd_MF, responses, info, freqs,
    )
    results_cluster = {
        "R":  dict(r_vals=r_R_cl,  p_corr=p_R_cl),
        "MF": dict(r_vals=r_MF_cl, p_corr=p_MF_cl),
    }
    _plot_topomap_grid(results_cluster, freqs, info, "cluster_perm", col_name)

    # ---- Save all r / p-values (band-level + bin-wise, uncorrected and
    #      every correction) to Excel ----
    save_r_p_values_to_excel(
        col_name, freqs, ch_names,
        r_R,  p_R,  p_R_corr_by_method,  r_R_cl,  p_R_cl,
        r_MF, p_MF, p_MF_corr_by_method, r_MF_cl, p_MF_cl,
        r_band_R,  p_band_R,  p_band_R_corr_by_method,
        r_band_MF, p_band_MF, p_band_MF_corr_by_method,
    )

    # ---- Scatter plots (use the same band-level r/p driving the topomaps) ----
    _plot_electrode_scatter_grid(psd_db, sub_with_eeg, responses,
                                 ch_names, col_name, "R",  "Resting",
                                 r_band_R,  p_band_R)
    _plot_electrode_scatter_grid(psd_db, sub_with_eeg, responses,
                                 ch_names, col_name, "MF", "Mindfulness",
                                 r_band_MF, p_band_MF)


def main():
    df = load_behavioral_table(BEHAVIORAL_FILE)
    candidate_ids = [s for s in ALL_SUBJECT_IDS
                     if subject_id_to_label(s) not in ALWAYS_EXCLUDE]
    psd_db, freqs, info = build_psd_database(candidate_ids)
    if not psd_db:
        raise RuntimeError("No subject PSDs were computed. Check FIF file paths.")

    for col in BEHAVIORAL_COLS:
        if col not in df.columns:
            print(f'\n[warn] Column "{col}" not found in {BEHAVIORAL_FILE} — skipped.')
            continue
        run_for_behavioral_column(df, col, psd_db, freqs, info)

    print("\nAll behavioral columns processed.")
    print(f"Figures saved in: {os.path.abspath(OUT_DIR)}")


if __name__ == "__main__":
    main()


--- Step 1: Computing relative PSD (mean-of-ratios) for all subjects ---


C:\Users\acadena\AppData\Local\Temp\ipykernel_24064\748578434.py:205: RuntimeWarning: This filename (Sujeto1_R_processed.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs_R  = mne.read_epochs(file_R,  preload=True, verbose=False)
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\mne\event.py:184: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  event_list.shape = (-1, 3)
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\mne\_fiff\tag.py:180: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data.shape = dims
C:\Users\acadena\AppData\Local\Temp\ipykernel_24064\748578434.py:206: RuntimeWarning: This filename (Sujeto1_MF_process

  [ok]   Subject 1  (n_freqs=248)


C:\Users\acadena\AppData\Local\Temp\ipykernel_24064\748578434.py:205: RuntimeWarning: This filename (Sujeto2_R_processed.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs_R  = mne.read_epochs(file_R,  preload=True, verbose=False)
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\mne\event.py:184: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  event_list.shape = (-1, 3)
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\mne\_fiff\tag.py:180: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data.shape = dims
C:\Users\acadena\AppData\Local\Temp\ipykernel_24064\748578434.py:206: RuntimeWarning: This filename (Sujeto2_MF_process

  [ok]   Subject 2  (n_freqs=248)


C:\Users\acadena\AppData\Local\Temp\ipykernel_24064\748578434.py:205: RuntimeWarning: This filename (Sujeto3_R_processed.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs_R  = mne.read_epochs(file_R,  preload=True, verbose=False)
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\mne\event.py:184: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  event_list.shape = (-1, 3)
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\mne\_fiff\tag.py:180: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data.shape = dims
C:\Users\acadena\AppData\Local\Temp\ipykernel_24064\748578434.py:206: RuntimeWarning: This filename (Sujeto3_MF_process

... (93 outputs left out)

c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\statsmodels\stats\multitest.py:164: RuntimeWarning: divide by zero encountered in log1p
  pvals_corrected = -np.expm1(ntests * np.log1p(-pvals))
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\statsmodels\stats\multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\statsmodels\stats\multitest.py:164: RuntimeWarning: divide by zero encountered in log1p
  pvals_corrected = -np.expm1(ntests * np.log1p(-pvals))
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\statsmodels\stats\multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))


  -> Cluster-based permutation (Resting)...
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 61
  -> Cluster-based permutation (Mindfulness)...
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 61
  saved: figures_correlation_fooof\AR4-MF_pearson_cluster_perm.png
  saved: figures_correlation_fooof\AR4-MF_r_p_values.xlsx
  saved: figures_correlation_fooof\AR4-MF_scatter_resting.png
  saved: figures_correlation_fooof\AR4-MF_scatter_mindfulness.png

Behavioral column: AR2-AR
  N subjects after exclusions + EEG availability: 22
  Subjects used: [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 36, 37]
  band-contribution table: figures_correlation_fooof\AR2-AR_band_contributions_per_electrode.csv
                              figures_correlation_fooof\AR2-AR_band_contributions_per_electrode.

c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\statsmodels\stats\multitest.py:164: RuntimeWarning: divide by zero encountered in log1p
  pvals_corrected = -np.expm1(ntests * np.log1p(-pvals))
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\statsmodels\stats\multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\statsmodels\stats\multitest.py:164: RuntimeWarning: divide by zero encountered in log1p
  pvals_corrected = -np.expm1(ntests * np.log1p(-pvals))
c:\Users\acadena\miniconda3\envs\MINDF\Lib\site-packages\statsmodels\stats\multitest.py:186: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(-pvals))


  -> Cluster-based permutation (Resting)...
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 61
  -> Cluster-based permutation (Mindfulness)...
Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.
-- number of adjacent vertices : 61
  saved: figures_correlation_fooof\AR2-AR_pearson_cluster_perm.png
  saved: figures_correlation_fooof\AR2-AR_r_p_values.xlsx
  saved: figures_correlation_fooof\AR2-AR_scatter_resting.png
  saved: figures_correlation_fooof\AR2-AR_scatter_mindfulness.png

All behavioral columns processed.
Figures saved in: c:\Users\acadena\Downloads\Topomaps\figures_correlation_fooof
